# 第 4 讲：Attention 替代方案与 MoE

这是 CS336 Lecture 04 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_04.pdf`
- 官方形式：PDF lecture
- 英文标题：Attention Alternatives And Mixtures Of Experts

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲围绕长上下文成本展开：标准 attention 是二次复杂度，于是出现 local attention、linear attention、state-space/recurrence 和 MoE 等方向。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

先看 attention 为什么贵，再看降低序列成本或增加参数容量的不同路线。


In [1]:
lecture = 4
title = 'Attention Alternatives And Mixtures Of Experts'
source = 'external/cs336-lectures/lecture_04.pdf'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 4: Attention Alternatives And Mixtures Of Experts
source: external/cs336-lectures/lecture_04.pdf


## 2. 标准 attention 的瓶颈

QK^T 需要 n^2 级别的 token-token 交互，长上下文下成本迅速上升。


## 3. Local / sparse attention

只看局部窗口或少量全局 token，用表达能力换效率。


## 4. Linear attention

通过改写或近似 attention，把二次 token 交互变成线性或近线性成本。


## 5. MoE

Mixture of Experts 用 router 选择少数 expert，让总参数变大但每个 token 只激活一部分。


## 6. 核心 tradeoff

这些方法都在表达能力、训练稳定性、硬件效率和工程复杂度之间做交换。


## 动手实验 1：Attention 成本随序列长度增长

先直接运行，再改一个数字或字符串。


In [2]:
for n in [1024, 4096, 16384, 65536]:
    full = n * n
    local = n * 512
    print(f"n={n:6d} full={full/1e6:10.1f}M local_window={local/1e6:8.1f}M")


n=  1024 full=       1.0M local_window=     0.5M
n=  4096 full=      16.8M local_window=     2.1M
n= 16384 full=     268.4M local_window=     8.4M
n= 65536 full=    4295.0M local_window=    33.6M


## 动手实验 2：Top-1 MoE router 玩具版

先直接运行，再改一个数字或字符串。


In [3]:
import random

random.seed(0)
num_tokens = 20
num_experts = 4
assignments = [random.randrange(num_experts) for _ in range(num_tokens)]
load = {e: assignments.count(e) for e in range(num_experts)}
print("assignments:", assignments)
print("expert load:", load)


assignments: [3, 3, 0, 2, 3, 3, 2, 3, 2, 1, 1, 2, 1, 0, 2, 1, 2, 0, 0, 2]
expert load: {0: 4, 1: 4, 2: 7, 3: 5}


## 动手实验 3：Load balancing 损失的直觉

先直接运行，再改一个数字或字符串。


In [4]:
loads = [8, 7, 4, 1]
ideal = sum(loads) / len(loads)
imbalance = sum((x - ideal) ** 2 for x in loads) / len(loads)
print("loads:", loads)
print("ideal:", ideal)
print("imbalance penalty:", imbalance)


loads: [8, 7, 4, 1]
ideal: 5.0
imbalance penalty: 7.5


## 今日检查点

1. 能解释标准 attention 为什么是 O(n^2)。
2. 能说出 local attention 省在哪里、损失在哪里。
3. 能解释 MoE 的 activated parameters 和 total parameters 的区别。
4. 能理解 router load balancing 为什么重要。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
